In [1]:
from references import zeng24_config, zeng24_question
from rag_components import vector_retriever, get_prompts, vector_retrieved_contexts, vector_embed_model, get_data_chunks
from tools import load_saved_data, eva_pub_pri_hitnum, eva_pii_hitnum, eva_repeat_context, eva_rouge, eva_bleu, eva_embedding_similarity
import torch
import os
from tqdm import tqdm
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

/workspace/zms/miniconda3/envs/rag-llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 09-24 10:16:49 [__init__.py:244] Automatically detected platform cuda.


In [2]:

def run_llm_translate(all_corpus_text):
    client = OpenAI(base_url="http://localhost:22999/v1", api_key="EMPTY")

    def call_api(prompt):
        response = client.chat.completions.create(
            model="./Models/Qwen2.5-32B-Instruct",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            top_p=0.8,
            max_tokens=1024,
        )
        return response.choices[0].message.content.strip()

    answers = []
    

    with ThreadPoolExecutor(max_workers=50) as executor:
        answers = list(executor.map(call_api, all_corpus_text))  # 顺序和输入一样

    # for prompt in tqdm(all_prompts, desc="Generating answers", unit="prompt"):
    #     response = client.chat.completions.create(
    #         model=cfg.llm.model_name,
    #         messages=[{"role": "user", "content": prompt}],
    #         temperature=cfg.llm.temperature,
    #         top_p=cfg.llm.top_p,
    #         max_tokens=cfg.llm.max_gen_len,
    #     )
    #     generated_text = response.choices[0].message.content.strip()
    #     answers.append(generated_text)

    return answers

In [3]:
cfg = zeng24_config.Zeng24fiqa()

In [4]:
import json

In [28]:
corpus = {}
for path in ["/workspace/zms/Data/rag-llm/data/fiqa/queries.jsonl"]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            doc = json.loads(line)
            corpus[doc["_id"]] = {
                "_id": doc["_id"],
                # "title": doc.get("title", ""),
                "text": doc.get("text", "")
            }

In [29]:
for corpus_i in tqdm(corpus.values()):
    print(corpus_i)
    break


  0%|          | 0/6648 [00:00<?, ?it/s]

{'_id': '0', 'text': 'What is considered a business expense on a business trip?'}


In [30]:
prompt_template = """Translate the following English text accurately and fluently into {target_language}.  
Requirements:
- Faithfully preserve the original meaning, tone, and style (e.g., formal, technical, conversational).
- Use natural, idiomatic expressions in {target_language}.
- Do not add explanations, comments, or extra content.
- Handle proper nouns, technical terms, or brand names according to standard conventions in {target_language}.

English original:
"{text_original}"

{target_language} translation:"""

In [31]:
client = OpenAI(base_url="http://localhost:22999/v1", api_key="EMPTY")

def call_api(prompt):
    response = client.chat.completions.create(
        model="./Models/Qwen2.5-32B-Instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        top_p=0.8,
        max_tokens=1024,
    )
    return response.choices[0].message.content.strip()

answers = []

In [32]:
for corpus_i in tqdm(corpus.values()):
    ans = call_api(prompt_template.format(target_language="Chinese", text_original=corpus_i["text"]))
    print(ans)
    break

  0%|          | 0/6648 [00:00<?, ?it/s]

什么是商务旅行中的业务费用？


In [33]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from openai import OpenAI

# Prompt 模板（普通字符串，安全）
PROMPT_TEMPLATE = """
Translate the following English text accurately and fluently into {target_language}.  
Requirements:
- Faithfully preserve the original meaning, tone, and style.
- Use natural, idiomatic expressions in {target_language}.
- Do not add explanations, comments, or extra content.
- Handle proper nouns and technical terms per {target_language} conventions.

English original:
"{text_original}"

{target_language} translation:
"""

def run_llm_translate_dict(corpus_dict, target_language="Chinese", max_workers=20):
    """
    将英文语料字典并行翻译为目标语言，返回相同结构的翻译字典。
    
    Args:
        corpus_dict (dict): 格式如 {"id1": {"text": "Hello"}, "id2": {"text": "AI is great"}}
        target_language (str): 目标语言名称，如 "Chinese", "French"
        max_workers (int): 并发线程数（建议 10~30，避免压垮本地服务）
    
    Returns:
        dict: {"id1": "你好", "id2": "人工智能很棒"}
    """
    if not isinstance(corpus_dict, dict):
        raise ValueError("Input must be a dictionary with 'text' in each value.")

    client = OpenAI(base_url="http://localhost:22999/v1", api_key="EMPTY")

    def call_api(prompt):
        response = client.chat.completions.create(
            model="./Models/Qwen2.5-32B-Instruct",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            top_p=0.8,
            max_tokens=1024,
        )
        return response.choices[0].message.content.strip()

    # 提取 keys 和 texts，保持顺序
    keys = list(corpus_dict.keys())
    texts = [corpus_dict[k]["text"] for k in keys]

    # 构建 prompts
    all_prompts = [
        PROMPT_TEMPLATE.format(
            target_language=target_language,
            text_original=text.replace('"', '\\"')  # 转义双引号，防止 JSON/Prompt 错乱
        )
        for text in texts
    ]

    # 并行翻译（带进度条）
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        translated_texts = list(tqdm(
            executor.map(call_api, all_prompts),
            total=len(all_prompts),
            desc=f"Translating to {target_language}",
            unit="item"
        ))

    # 重建字典（保持原始 key 顺序）
    result_dict = {key: trans_text for key, trans_text in zip(keys, translated_texts)}
    return result_dict

In [34]:
# 调用翻译函数
translated_dict = run_llm_translate_dict(
    corpus_dict=corpus,
    target_language="Chinese",
    max_workers=50
)

Translating to Chinese: 100%|██████████| 6648/6648 [01:38<00:00, 67.52item/s] 


In [35]:
translated_dict

{'0': '"商务出差中被视为业务费用的项目是什么？"',
 '4': '"商务费用 - 商务出差期间发生事故的汽车保险免赔额"',
 '5': '"开启新的在线业务"',
 '6': '“营业日”和“到期日”对于账单',
 '7': '新业务所有者——企业的税收和个人的税收有何不同？',
 '9': '"爱好与生意"',
 '11': '"个人支票而非商业支票"',
 '12': '美国税法是否要求小企业主将商业采购计入个人收入？',
 '13': '如何在不提供业务地址的情况下注册英国公司？',
 '14': '什么是“商业基础”？',
 '16': '"来自前一年的业务投资损失"',
 '19': '如何估算一个收入为零的企业的税款/报税费用？',
 '20': '通过商业贷款为企业购买汽车是否被视为业务支出？',
 '21': '"扣除去年（未记录的）副业损失"',
 '23': '"30%的业务股份"',
 '25': '"从个人信用卡报销商务费用"',
 '27': '使用商业支票在零售店付款',
 '28': '"在全职工作的同时创业时的税务问题"',
 '30': '我可以还清信用卡余额来释放可用信用额度吗？',
 '31': '“慢慢开始副业”',
 '32': '为什么“现金支票兑换”是一项合法的生意？',
 '33': '成为百万富翁只有做生意这一条路吗？',
 '35': '"评估一家小企业以进行投资"',
 '36': '“用不算大的收入启动一个大生意？”',
 '37': '"申报营业税的要求？"',
 '41': '"为工作和我的生意准备的个人退休账户（IRA）"',
 '43': '关于汇款业务的建议',
 '45': '如何将个人汽车租赁转为商业汽车租赁？',
 '46': '"辅导业务薪酬管理"',
 '47': '作为小企业主，我应该从个人支票账户还是企业支票账户支付税款？',
 '48': '"我丈夫的公司应该支付我的生意费用吗？"',
 '49': '为什么在线交易不能在非营业时间完成？',
 '51': '全职工作+经营小副业：税收的最佳业务结构？',
 '52': '新的自动售货路线业务，不确定如何确定税收',
 '53': '“寻找一个好的小型企业会计师？”',
 '54': '"全职工作应税收入 + 商业收益"',

In [36]:
for key, value in list(translated_dict.items()):
    corpus[key]["text"] = value
    # print(f"{key}: {value}\n")

In [37]:
# 指定输出文件名
output_file = 'output.jsonl'

# 写入文件：每行一个 JSON 对象
with open(output_file, 'w', encoding='utf-8') as f:
    for key in sorted(corpus.keys(), key=lambda x: int(x)):  # 按 _id 数值排序（可选）
        json_record = corpus[key]
        # 确保写入的是标准 JSON（不带额外空格，ensure_ascii=False 支持中文）
        f.write(json.dumps(json_record, ensure_ascii=False) + '\n')